# 🚀 AdamV: CIFAR-10 Grand Finale Benchmark
This notebook tests the real-world computer vision performance of **AdamV** vs **AdamW**.
We train a **ResNet-9** architecture on the official CIFAR-10 dataset.
- **AdamW** uses CosineAnnealingLR.
- **AdamV** uses its native Ramanujan Envelope and OMNI-ModBH.

In [ ]:
!pip install ninja matplotlib torchvision


In [ ]:

import os
import torch
from torch.utils.cpp_extension import load_inline

# Configure CUDA for T4 GPUs on Kaggle
os.environ['TORCH_CUDA_ARCH_LIST'] = "7.5"
os.environ['NVIDIA_VISIBLE_DEVICES'] = "all"
os.environ['OMP_NUM_THREADS'] = "1"

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

# Include CUDAContext to fix stream issues
cpp_source = """
#include <torch/extension.h>
#include <ATen/cuda/CUDAContext.h>

void adamv_step_cuda(at::Tensor p, at::Tensor grad, at::Tensor exp_avg, at::Tensor exp_avg_sq, at::Tensor direcao, float lr, float beta1, float beta2, float eps, float weight_decay, float progresso, float bakh_thresh_eff, int step, int D, bool omni_triggered, int64_t punning_mask);
"""

cuda_source = """
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <cmath>
#include <ATen/cuda/CUDAContext.h>

#ifndef M_PI
#define M_PI 3.14159265358979323846
#endif

const int BLOCK_SIZE = 256;

template <typename scalar_t>
__global__ void adamv_prepare_kernel(
    float* __restrict__ exp_avg,
    float* __restrict__ exp_avg_sq,
    float* __restrict__ direcao_buffer,
    const scalar_t* __restrict__ grad,
    float beta1, float beta2, float bias_correction1, float bias_correction2, float eps, int numel) {
    
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < numel) {
        float g = static_cast<float>(grad[idx]);
        float m = exp_avg[idx];
        float v = exp_avg_sq[idx];

        // BRCM: Bakhshali Residual-Coupled Momentum
        float sqrt_v = sqrt(v);
        float bakh_residual = (g * g) / (2.0f * (sqrt_v + std::abs(g) + eps));
        float curvature_shift = bakh_residual / (sqrt_v + eps);
        float beta1_dynamic = beta1 * std::exp(-curvature_shift);
        
        m = beta1_dynamic * m + (1.0f - beta1_dynamic) * g;
        v = beta2 * v + (1.0f - beta2) * g * g;
        
        exp_avg[idx] = m;
        exp_avg_sq[idx] = v;

        float m_hat = m / bias_correction1;
        float v_hat = v / bias_correction2;
        
        direcao_buffer[idx] = m_hat / (sqrt(v_hat) + eps);
    }
}

template <typename scalar_t>
__global__ void adamv_update_kernel(
    scalar_t* __restrict__ params,
    const scalar_t* __restrict__ grad,
    const float* __restrict__ exp_avg_sq,
    const float* __restrict__ direcao_buffer,
    const float* __restrict__ norm_tensor_ptr,
    float progresso, float cooling_factor, float bakh_thresh_eff, float bias_correction2, float eps, 
    float wd_factor, float lr_max, float weight_decay, int numel, int D,
    bool omni_triggered, uint32_t punning_mask) {
    
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < numel) {
        scalar_t p = params[idx];
        
        float g = static_cast<float>(grad[idx]);
        float dir = direcao_buffer[idx];
        float v = exp_avg_sq[idx];
        
        float norm_dir = (*norm_tensor_ptr) / sqrt(static_cast<float>(D));
        float envelope = (1.0f + progresso) / (progresso + norm_dir + eps);
        float lr_efetivo = lr_max * min(envelope * cooling_factor, 1.5f);
        
        float a = lr_efetivo * dir;
        float v_hat = v / bias_correction2;
        float sqrt_v = sqrt(v_hat);
        
        bool explosao_mask = std::abs(g) > (bakh_thresh_eff * sqrt_v);
        
        // Bakhshali Quartic Brake
        float denom = sqrt_v + std::abs(a) + eps;
        float correction = (a * a) / (2.0f * denom);
        float bakhshali_brake = a - copysignf(1.0f, a) * correction;
        
        float step_size = explosao_mask ? bakhshali_brake : a;
        
        if (weight_decay != 0.0f) {
            p = static_cast<scalar_t>(static_cast<float>(p) * (1.0f - lr_max * weight_decay * wd_factor));
        }
        
        params[idx] = static_cast<scalar_t>(static_cast<float>(p) - step_size);
        
        if (omni_triggered && sizeof(scalar_t) == 4) {
            float p_new = static_cast<float>(params[idx]);
            uint32_t p_int = __float_as_uint(p_new);
            
            uint32_t sign = p_int & 0x80000000;
            uint32_t exp  = p_int & 0x7F800000;
            uint32_t mant = p_int & 0x007FFFFF;
            
            uint32_t mant_mod = (((mant + 1) * 31337) & 0x007FFFFF) & punning_mask;
            
            p_new = __uint_as_float(sign | exp | mant_mod);
            params[idx] = static_cast<scalar_t>(p_new);
        }
    }
}

#define CHECK_CUDA(x) TORCH_CHECK(x.device().is_cuda(), #x " must be a CUDA tensor")
#define CHECK_CONTIGUOUS(x) TORCH_CHECK(x.is_contiguous(), #x " must be contiguous")
#define CHECK_INPUT(x) CHECK_CUDA(x); CHECK_CONTIGUOUS(x)

void adamv_step_cuda(
    at::Tensor p,
    at::Tensor grad,
    at::Tensor exp_avg,
    at::Tensor exp_avg_sq,
    at::Tensor direcao,
    float lr,
    float beta1,
    float beta2,
    float eps,
    float weight_decay,
    float progresso,
    float bakh_thresh_eff,
    int step,
    int D,
    bool omni_triggered,
    int64_t punning_mask) 
{
    CHECK_INPUT(p);
    CHECK_INPUT(grad);
    CHECK_INPUT(exp_avg);
    CHECK_INPUT(exp_avg_sq);
    CHECK_INPUT(direcao);

    int numel = p.numel();
    int blocks = (numel + BLOCK_SIZE - 1) / BLOCK_SIZE;

    float bias_correction1 = 1.0f - std::pow(static_cast<float>(beta1), static_cast<float>(step));
    float bias_correction2 = 1.0f - std::pow(static_cast<float>(beta2), static_cast<float>(step));

    cudaStream_t stream = at::cuda::getCurrentCUDAStream();

    AT_DISPATCH_FLOATING_TYPES_AND_HALF(p.scalar_type(), "adamv_prepare", [&] {
        adamv_prepare_kernel<scalar_t><<<blocks, BLOCK_SIZE, 0, stream>>>(
            exp_avg.data_ptr<float>(),
            exp_avg_sq.data_ptr<float>(),
            direcao.data_ptr<float>(),
            grad.data_ptr<scalar_t>(),
            beta1, beta2, bias_correction1, bias_correction2, eps, numel
        );
    });

    // Compute norm asynchronously on GPU
    at::Tensor norm_tensor = at::linalg_norm(direcao);
    
    float cooling_factor = 0.5f * (1.0f + std::cos(M_PI * progresso));
    float wd_factor = 0.5f * (1.0f + std::cos(M_PI * progresso));

    AT_DISPATCH_FLOATING_TYPES_AND_HALF(p.scalar_type(), "adamv_update", [&] {
        adamv_update_kernel<scalar_t><<<blocks, BLOCK_SIZE, 0, stream>>>(
            p.data_ptr<scalar_t>(),
            grad.data_ptr<scalar_t>(),
            exp_avg_sq.data_ptr<float>(),
            direcao.data_ptr<float>(),
            norm_tensor.data_ptr<float>(),
            progresso, cooling_factor, bakh_thresh_eff, bias_correction2, eps, wd_factor, lr, weight_decay, numel, D,
            omni_triggered, static_cast<uint32_t>(punning_mask)
        );
    });
}

"""

print("Compiling AdamV CUDA Kernel (JIT) ...")
adamv_cuda = load_inline(
    name='adamv_cuda',
    cpp_sources=cpp_source,
    cuda_sources=cuda_source,
    functions=['adamv_step_cuda'],
    with_cuda=True,
    extra_cflags=['-O3', '-fopenmp'],
    extra_cuda_cflags=['-O3', '-use_fast_math', '-lineinfo'],
    verbose=True
)
print("CUDA Kernel loaded successfully!")

class DummyCPU:
    pass
adamv_cpp = DummyCPU()


In [ ]:
import torch
import math

class AdamV(torch.optim.Optimizer):
    """
    AdamV (Adam Vedic) Optimizer - Pure Python Version.
    AdamV 3.1: Harmonic Refactor (In-Place VRAM Opt, OMNI State Fix, Modular Flags)
    """
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, 
                 weight_decay=0.01, total_steps=10000, 
                 bakhshali_threshold=10.0, enable_omni=True,
                 lp_kappa=0.1, lp_omega=10.0, punning_mask=0xFFFFE000,
                 enable_ignition=True, enable_cooling=False, enable_brake=True):
                 
        if not 0.0 <= lr:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay,
                        total_steps=total_steps, bakhshali_threshold=bakhshali_threshold,
                        enable_omni=enable_omni, lp_kappa=lp_kappa, lp_omega=lp_omega, 
                        punning_mask=punning_mask,
                        enable_ignition=enable_ignition, enable_cooling=enable_cooling, enable_brake=enable_brake)
        super(AdamV, self).__init__(params, defaults)
        
        self.state['omni_global'] = {
            'loss_ema': float('inf'),
            'patience': 0,
            'clock_reset_step': 0,
            'global_step': 0,
        }

    @torch.no_grad()
    def step(self, closure=None, current_loss=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
                
        if current_loss is not None:
            loss = current_loss

        g_state = self.state['omni_global']
        g_state['global_step'] += 1
        current_step = g_state['global_step']
        
        omni_triggered = False
        if loss is not None and len(self.param_groups) > 0 and self.param_groups[0]['enable_omni']:
            loss_val = float(loss) if isinstance(loss, torch.Tensor) else loss
            if g_state['loss_ema'] == float('inf'):
                g_state['loss_ema'] = loss_val
                g_state['patience'] = 0
            else:
                g_state['loss_ema'] = 0.9 * g_state['loss_ema'] + 0.1 * loss_val
                
            is_worse = loss_val > g_state['loss_ema'] * 0.99
            g_state['patience'] = g_state['patience'] + 1 if is_worse else 0
            
            patience_limit = max(500, int(self.param_groups[0]['total_steps'] * 0.05))
            if g_state['patience'] >= patience_limit:
                omni_triggered = True
                g_state['patience'] = 0
                g_state['clock_reset_step'] = current_step
                
        for group in self.param_groups:
            lr_max = group['lr']
            total_steps = group['total_steps']
            
            # Autonomous Ignition
            if group.get('enable_ignition', True):
                ignition = min(1.0, current_step / max(1.0, total_steps * 0.10))
                lr_max = lr_max * ignition
                
            beta1, beta2 = group['betas']
            eps = group['eps']
            weight_decay = group['weight_decay']
            total_steps = group['total_steps']
            bakh_thresh = group['bakhshali_threshold']
            lp_kappa = group['lp_kappa']
            lp_omega = group['lp_omega']
            punning_mask = group['punning_mask']
            
            internal_step = current_step - g_state['clock_reset_step']
            progresso = min(1.0, internal_step / max(1, total_steps))
            
            LP_Fator = 1.0 + lp_kappa * math.cos(lp_omega * math.log(1.0 + progresso * 10.0))
            bakh_thresh_eff = bakh_thresh * LP_Fator
            
            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p, memory_format=torch.preserve_format, dtype=torch.float32)
                    state['exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format, dtype=torch.float32)
                
                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                state['step'] += 1
                
                # In-Place Momentum Update
                snr = (exp_avg * exp_avg) / (exp_avg_sq + eps)
                beta1_eff = torch.clamp(beta1 + 0.05 * (1.0 - 2.0 * snr), 0.0, 1.0)
                exp_avg.mul_(beta1_eff).add_(grad.float(), alpha=1.0 - beta1_eff)
                
                exp_avg_sq.mul_(beta2).addcmul_(grad.float(), grad.float(), value=1.0 - beta2)
                
                bias_correction1 = 1 - beta1 ** state['step']
                bias_correction2 = 1 - beta2 ** state['step']
                
                sqrt_v = exp_avg_sq.sqrt().div_(math.sqrt(bias_correction2))
                direcao = (exp_avg / bias_correction1) / (sqrt_v + eps)
                
                norm_dir_padrao = torch.linalg.norm(direcao) / math.sqrt(p.numel())
                
                if group.get('enable_cooling', False):
                    envelope = (1.0 + progresso) / (progresso + norm_dir_padrao + eps)
                    cooling_factor = 0.5 * (1.0 + math.cos(math.pi * progresso))
                    lr_efetivo = lr_max * torch.clamp(envelope * cooling_factor, max=1.5)
                else:
                    lr_efetivo = lr_max
                
                a = direcao.mul_(lr_efetivo)
                
                explosao_mask = torch.abs(grad) > (bakh_thresh_eff * sqrt_v)
                denom = sqrt_v.add_(torch.abs(a)).add_(eps)
                correction = (a * a).div_(denom.mul_(2.0))
                
                bakhshali_brake = a.clone().sub_(torch.sign(a) * correction)
                
                if group.get('enable_brake', True):
                    step_size = torch.where(explosao_mask, bakhshali_brake, a)
                else:
                    step_size = a
                
                if weight_decay != 0:
                    wd_factor = 0.5 * (1.0 + math.cos(math.pi * progresso))
                    p.mul_(1.0 - lr_max * weight_decay * wd_factor)
                    
                p.sub_(step_size)
                
                if omni_triggered:
                    # Deterministic Mantissa Teleportation (Zero-RAM escape)
                    p_int = p.view(torch.int32)
                    sign_exp = p_int & 0xFF800000
                    mant = p_int & 0x007FFFFF
                    
                    mask_val = int(punning_mask)
                    if mask_val > 0x7FFFFFFF:
                        mask_val -= 0x100000000
                    
                    conditional_mask = mask_val if p.dtype == torch.float32 else -1
                    scrambled_mant = (((mant + 1) * 31337) & 0x007FFFFF) & conditional_mask
                    
                    p_new = (sign_exp | scrambled_mant).view(torch.float32)
                    p.copy_(p_new)
                    
                    # Harmonic State Reset: Do not poison momentum. Flush it gracefully.
                    state['exp_avg'].zero_()
                    state['exp_avg_sq'].mul_(0.1)
                
        return loss

class AdamVCpp(torch.optim.Optimizer):
    """
    AdamV (Adam Vedic) Optimizer - C++ Fused Kernel Version.
    AdamV 3.1: Harmonic Refactor
    """
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, 
                 weight_decay=0.01, total_steps=10000, 
                 bakhshali_threshold=10.0, enable_omni=True,
                 lp_kappa=0.1, lp_omega=10.0, punning_mask=0xFFFFE000,
                 enable_ignition=True, enable_cooling=False, enable_brake=True):
                 
        try:
            pass # import adamv_cpp
            self.adamv_cpp = adamv_cpp
        except ImportError:
            raise ImportError("AdamVCpp requer que a extensao C++ seja compilada")
            
        try:
            pass # import adamv_cuda
            self.adamv_cuda = adamv_cuda
        except ImportError:
            self.adamv_cuda = None
            
        if not 0.0 <= lr:
            raise ValueError(f"Invalid learning rate: {lr}")
            
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay,
                        total_steps=total_steps, bakhshali_threshold=bakhshali_threshold,
                        enable_omni=enable_omni, lp_kappa=lp_kappa, lp_omega=lp_omega, 
                        punning_mask=punning_mask,
                        enable_ignition=enable_ignition, enable_cooling=enable_cooling, enable_brake=enable_brake)
        super(AdamVCpp, self).__init__(params, defaults)
        
        self.state['omni_global'] = {
            'loss_ema': float('inf'),
            'patience': 0,
            'clock_reset_step': 0,
            'global_step': 0,
        }

    @torch.no_grad()
    def step(self, closure=None, current_loss=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        if current_loss is not None:
            loss = current_loss

        g_state = self.state['omni_global']
        g_state['global_step'] += 1
        current_step = g_state['global_step']
        
        omni_triggered = False
        if loss is not None and len(self.param_groups) > 0 and self.param_groups[0]['enable_omni']:
            loss_val = float(loss) if isinstance(loss, torch.Tensor) else loss
            if g_state['loss_ema'] == float('inf'):
                g_state['loss_ema'] = loss_val
                g_state['patience'] = 0
            else:
                g_state['loss_ema'] = 0.9 * g_state['loss_ema'] + 0.1 * loss_val
                
            is_worse = loss_val > g_state['loss_ema'] * 0.99
            g_state['patience'] = g_state['patience'] + 1 if is_worse else 0
            
            patience_limit = max(500, int(self.param_groups[0]['total_steps'] * 0.05))
            if g_state['patience'] >= patience_limit:
                omni_triggered = True
                g_state['patience'] = 0
                g_state['clock_reset_step'] = current_step
                
        for group in self.param_groups:
            lr_max = group['lr']
            total_steps = group['total_steps']
            
            if group.get('enable_ignition', True):
                ignition = min(1.0, current_step / max(1.0, total_steps * 0.10))
                lr_max = lr_max * ignition
                
            beta1, beta2 = group['betas']
            eps = group['eps']
            weight_decay = group['weight_decay']
            total_steps = group['total_steps']
            bakh_thresh = group['bakhshali_threshold']
            lp_kappa = group['lp_kappa']
            lp_omega = group['lp_omega']
            punning_mask = group['punning_mask']
            
            internal_step = current_step - g_state['clock_reset_step']
            progresso = min(1.0, internal_step / max(1, total_steps))
            
            LP_Fator = 1.0 + lp_kappa * math.cos(lp_omega * math.log(1.0 + progresso * 10.0))
            bakh_thresh_eff = bakh_thresh * LP_Fator
            
            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p, memory_format=torch.preserve_format, dtype=torch.float32)
                    state['exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format, dtype=torch.float32)
                    state['direcao_buffer'] = torch.empty_like(p, memory_format=torch.preserve_format, dtype=torch.float32)
                
                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                state['step'] += 1
                
                # C++ Fused Kernel Call
                # OMNI logic is pushed to the END inside the C++ Kernel now
                mask_val = int(punning_mask)
                if mask_val > 0x7FFFFFFF:
                    mask_val -= 0x100000000
                    
                if p.is_cpu:
                    self.adamv_cpp.adamv_step_cpu(
                        p, grad, exp_avg, exp_avg_sq, state['direcao_buffer'],
                        lr_max, beta1, beta2, eps, weight_decay,
                        float(progresso), float(bakh_thresh_eff), state['step'], p.numel()
                    )
                elif p.is_cuda and self.adamv_cuda is not None and hasattr(self.adamv_cuda, 'adamv_step_cuda'):
                    self.adamv_cuda.adamv_step_cuda(
                        p, grad, exp_avg, exp_avg_sq, state['direcao_buffer'],
                        lr_max, beta1, beta2, eps, weight_decay,
                        float(progresso), float(bakh_thresh_eff), state['step'], p.numel(),
                        bool(omni_triggered), mask_val
                    )
                else:
                    # Python fallback para GPU - 100% In-Place BRCM
                    bias_correction1 = 1 - beta1 ** state['step']
                    bias_correction2 = 1 - beta2 ** state['step']
                    
                    sqrt_v = exp_avg_sq.sqrt()
                    denom_brcm = sqrt_v.clone().add_(torch.abs(grad.float())).add_(eps)
                    bakh_residual = (grad.float() * grad.float()).div_(denom_brcm.mul_(2.0))
                    curvature_shift = bakh_residual.div_(sqrt_v.add_(eps))
                    
                    beta1_eff = torch.exp(-curvature_shift).mul_(beta1)
                    
                    # Update momentum In-Place
                    exp_avg.mul_(beta1_eff).add_(grad.float() * (1.0 - beta1_eff))
                    exp_avg_sq.mul_(beta2).addcmul_(grad.float(), grad.float(), value=1 - beta2)
                    
                    m_hat = exp_avg / bias_correction1
                    v_hat = exp_avg_sq / bias_correction2
                    direcao = m_hat / (v_hat.sqrt() + eps)
                    
                    norm_dir = torch.linalg.norm(direcao) / math.sqrt(p.numel())
                    if group.get('enable_cooling', False):
                        envelope = (1.0 + progresso) / (progresso + norm_dir + eps)
                        cooling = 0.5 * (1.0 + math.cos(math.pi * progresso))
                        lr_efetivo = lr_max * torch.clamp(envelope * cooling, max=1.5)
                    else:
                        lr_efetivo = lr_max
                    
                    a = direcao.mul_(lr_efetivo)
                    sqrt_v_hat = v_hat.sqrt()
                    
                    explosao_mask = torch.abs(grad) > (bakh_thresh_eff * sqrt_v_hat)
                    
                    denom = sqrt_v_hat.add_(torch.abs(a)).add_(eps)
                    correction = (a * a).div_(denom.mul_(2.0))
                    bakhshali_brake = a.clone().sub_(torch.sign(a) * correction)
                    
                    if group.get('enable_brake', True):
                        step_size = torch.where(explosao_mask, bakhshali_brake, a)
                    else:
                        step_size = a
                        
                    if weight_decay != 0:
                        wd_factor = 0.5 * (1.0 + math.cos(math.pi * progresso))
                        p.mul_(1.0 - lr_max * weight_decay * wd_factor)
                        
                    p.sub_(step_size)
                    
                    if omni_triggered:
                        p_int = p.view(torch.int32)
                        sign_exp = p_int & 0xFF800000
                        mant = p_int & 0x007FFFFF
                        
                        conditional_mask = mask_val if p.dtype == torch.float32 else -1
                        scrambled_mant = (((mant + 1) * 31337) & 0x007FFFFF) & conditional_mask
                        
                        p_new = (sign_exp | scrambled_mant).view(torch.float32)
                        p.copy_(p_new)
                        
                        state['exp_avg'].zero_()
                        state['exp_avg_sq'].mul_(0.1)
                        
        return loss


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import time
import os
import sys

# Ensure AdamV can be imported
# sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
try:
    pass
except ImportError:
    print("Warning: Could not import AdamVCpp. Ensure you are running from the correct directory.")

class Mul(nn.Module):
    def __init__(self, weight):
        super(Mul, self).__init__()
        self.weight = weight
    def forward(self, x):
        return x * self.weight

class Flatten(nn.Module):
    def forward(self, x): return x.view(x.size(0), -1)

class Residual(nn.Module):
    def __init__(self, module):
        super(Residual, self).__init__()
        self.module = module
    def forward(self, x): return x + self.module(x)

def conv_bn(channels_in, channels_out, kernel_size=3, stride=1, padding=1, groups=1):
    return nn.Sequential(
            nn.Conv2d(channels_in, channels_out, kernel_size=kernel_size,
                      stride=stride, padding=padding, groups=groups, bias=False),
            nn.BatchNorm2d(channels_out),
            nn.ReLU(inplace=True)
    )

class ResNet9(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNet9, self).__init__()
        self.model = nn.Sequential(
            conv_bn(3, 64, kernel_size=3, stride=1, padding=1),
            conv_bn(64, 128, kernel_size=5, stride=2, padding=2),
            Residual(nn.Sequential(conv_bn(128, 128), conv_bn(128, 128))),
            conv_bn(128, 256, kernel_size=3, stride=1, padding=1),
            nn.MaxPool2d(2),
            Residual(nn.Sequential(conv_bn(256, 256), conv_bn(256, 256))),
            conv_bn(256, 128, kernel_size=3, stride=1, padding=0),
            nn.AdaptiveMaxPool2d((1, 1)),
            Flatten(),
            nn.Linear(128, num_classes, bias=False),
            Mul(0.2)
        )
        
    def forward(self, x):
        return self.model(x)

def run_cifar10_arena():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running CIFAR-10 Arena on {device}")
    
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=512, shuffle=True, num_workers=2)
    
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
    testloader = torch.utils.data.DataLoader(testset, batch_size=512, shuffle=False, num_workers=2)
    
    optimizers_to_run = ['AdamW', 'AdamV']
    results = {}
    epochs = 15
    
    import gc
    
    for opt_name in optimizers_to_run:
        torch.cuda.empty_cache()
        gc.collect()
        torch.manual_seed(42)
        model = ResNet9().to(device)
        
        total_steps = epochs * len(trainloader)
        
        if opt_name == 'AdamW':
            opt = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=1e-4)
        else:
            opt = AdamVCpp(model.parameters(), lr=0.01, weight_decay=1e-4, total_steps=total_steps, bakhshali_threshold=10.0, enable_omni=False)
        
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_steps)
        criterion = nn.CrossEntropyLoss()
        
        train_losses = []
        test_accs = []
        
        print(f"\n[{opt_name}] Starting training...")
        start_time = time.time()
        
        for epoch in range(epochs):
            model.train()
            epoch_loss = 0.0
            
            for batch_idx, (inputs, targets) in enumerate(trainloader):
                inputs, targets = inputs.to(device), targets.to(device)
                
                opt.zero_grad(set_to_none=True)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                loss.backward()
                
                if opt_name == 'AdamW':
                    opt.step()
                else:
                    opt.step(current_loss=loss.item())
                scheduler.step()
                epoch_loss += loss.item()
                
            avg_train_loss = epoch_loss / len(trainloader)
            train_losses.append(avg_train_loss)
            
            # Validation
            model.eval()
            correct = 0
            total = 0
            with torch.no_grad():
                for inputs, targets in testloader:
                    inputs, targets = inputs.to(device), targets.to(device)
                    outputs = model(inputs)
                    _, predicted = outputs.max(1)
                    total += targets.size(0)
                    correct += predicted.eq(targets).sum().item()
            
            acc = 100. * correct / total
            test_accs.append(acc)
            print(f"[{opt_name}] Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Test Acc: {acc:.2f}%")
            
        total_time = time.time() - start_time
        print(f"[{opt_name}] Finished in {total_time:.2f}s | Final Acc: {test_accs[-1]:.2f}%")
        results[opt_name] = {'loss': train_losses, 'acc': test_accs}
        
    # Plot results
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    for name, data in results.items():
        ax1.plot(data['loss'], label=name, marker='o')
        ax2.plot(data['acc'], label=name, marker='o')
        
    ax1.set_title('CIFAR-10 Train Loss (ResNet-9)')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    ax2.set_title('CIFAR-10 Test Accuracy (ResNet-9)')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('cifar10_arena_results.png', dpi=200)
    print("\nResults saved to cifar10_arena_results.png")

if False:
    run_cifar10_arena()


In [ ]:
run_cifar10_arena()